# Velib exploration

Explore what is inside `data_1/`: which stations are active, how often they change, and how
free docks and bikes evolve over the day on a map.

Inputs (run the scripts first):
- `data_1/snapshots.csv` from `to_csv.py` (one row per snapshot and station)
- `data_1/stations.csv` from `build_stations.py` (name, coordinates, address, capacities)

Sections: 1. Load, 2. Overview, 3. Data quality, 4. Active stations, 5. How often stations change, 6. Map over time.

## 1. Load

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA = Path("data_1")
TZ = "Europe/Paris"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

In [ ]:
snap = pd.read_csv(
    DATA / "snapshots.csv",
    dtype={"src_ts": "int64", "station_id": "int64", "bikes": "int16", "docks": "int16",
           "mechanical": "float32", "ebike": "float32", "last_reported": "int64"},
)
stations = pd.read_csv(
    DATA / "stations.csv", encoding="utf-8",
    dtype={"station_code": "string", "house_number": "string", "street": "string",
           "postal_code": "string", "city": "string"},
)
# src_ts is a unix time (feed timestamp); show everything in Paris time
snap["time"] = pd.to_datetime(snap["src_ts"], unit="s", utc=True).dt.tz_convert(TZ)
print(f"{len(snap):,} rows, {snap.station_id.nunique()} stations")
snap.head()

In [ ]:
stations.head()

## 2. Overview

In [ ]:
times = snap[["src_ts", "time"]].drop_duplicates("src_ts").sort_values("src_ts").reset_index(drop=True)
times["gap_s"] = times["src_ts"].diff()
print(f"snapshots: {len(times)}")
print(f"from {times.time.iloc[0]:%a %d %b %H:%M} to {times.time.iloc[-1]:%a %d %b %H:%M} (Paris time)")
print(f"typical spacing: median {times.gap_s.median():.0f}s")
print("\ngaps longer than 5 minutes (missing data, e.g. laptop offline):")
big = times[times.gap_s > 300].copy()
big["gap_min"] = (big.gap_s / 60).round(1)
big[["time", "gap_min"]]

## 3. Data quality

Known quirks of the feed:
- **Glitch snapshots:** about once an hour the feed drops the mechanical/ebike split for all stations
  (`mechanical`/`ebike` empty). `bikes` and `docks` are still valid.
- **Capacity gap:** at some stations `bikes + docks` exceeded the official capacity, so one of the two is off.
- **Offline stations:** a few stations report 0 bikes and 0 docks in every snapshot, and some ids in the snapshots are
  missing from `stations.csv` when a station appeared after it was built (re-run `build_stations.py`). Both are flagged
  below and in section 4.

In [ ]:
frac_missing_types = snap.assign(no_types=snap.mechanical.isna()).groupby("src_ts")["no_types"].mean()
glitch_ts = frac_missing_types[frac_missing_types == 1].index
print(f"glitch snapshots (no mechanical/ebike for any station): {len(glitch_ts)}")
print(f"snapshots with only some stations missing the split: {((frac_missing_types > 0) & (frac_missing_types < 1)).sum()}")
times[times.src_ts.isin(glitch_ts)][["time"]]

In [ ]:
missing = sorted(set(snap.station_id) - set(stations.station_id))
print(f"stations in snapshots.csv but not in stations.csv (no name, dropped from the map): {missing}")
print("stations where we saw more bikes + docks than the official capacity:", (stations.capacity_gap < 0).sum())
stations.loc[stations.capacity_gap < 0].sort_values("capacity_gap")[
    ["station_id", "name", "city", "capacity_official", "observed_capacity", "capacity_gap"]
].head(10)

## 4. Which stations are active?

For each station and consecutive pair of snapshots, `changed` is true if `bikes` or `docks` differ from
the previous snapshot. The first snapshot of each station counts as not changed, and so does the first snapshot after a data gap (more than 5 minutes since the previous one, e.g. the laptop was offline): that step is not a one-minute change.
Note that the status flags (`is_installed`, `is_renting`, `is_returning`) are not in the CSV yet.

In [ ]:
snap = snap.sort_values(["station_id", "src_ts"]).reset_index(drop=True)
d = snap.groupby("station_id")[["bikes", "docks"]].diff()
# a step that crosses a data gap (e.g. laptop offline) is not a one-minute change: ignore it
GAP_S = 300
after_gap = snap.groupby("station_id")["src_ts"].diff() > GAP_S
d.loc[after_gap, :] = np.nan
print(f"steps ignored because they cross a data gap: {after_gap.sum()} ({snap.loc[after_gap, 'src_ts'].nunique()} snapshot(s))")
# false for a station's first snapshot and right after a gap: there is no one-minute step to measure there
snap["valid_step"] = d["bikes"].notna()
snap["d_bikes"] = d["bikes"].fillna(0)
snap["changed"] = (d["bikes"].fillna(0) != 0) | (d["docks"].fillna(0) != 0)
snap["abs_d_bikes"] = snap["d_bikes"].abs()
snap["is_empty"] = snap["bikes"].eq(0)
snap["is_full"] = snap["docks"].eq(0)

act = snap.groupby("station_id").agg(
    n_changes=("changed", "sum"),
    bikes_moved=("abs_d_bikes", "sum"),
    min_bikes=("bikes", "min"),
    max_bikes=("bikes", "max"),
    max_docks=("docks", "max"),
    pct_empty=("is_empty", "mean"),
    pct_full=("is_full", "mean"),
).join(stations.set_index("station_id")[["name", "city", "postal_code", "capacity_official"]])
act.describe().round(2)

In [ ]:
# a station with 0 bikes and 0 docks in every snapshot is offline (or has no slots): it would otherwise count as
# "always empty", "always full" and "never changed" at once, so it is counted separately
offline = (act.max_bikes == 0) & (act.max_docks == 0)
live = act[~offline]
summary = {
    "offline (always 0 bikes and 0 docks)": offline.sum(),
    "never changed": (live.n_changes == 0).sum(),
    "always empty (0 bikes)": (live.pct_empty == 1).sum(),
    "always full (0 docks)": (live.pct_full == 1).sum(),
    "empty >50% of the time": (live.pct_empty > 0.5).sum(),
    "full >50% of the time": (live.pct_full > 0.5).sum(),
}
print("offline stations:", act.index[offline].tolist())
pd.Series(summary, name="stations")

In [ ]:
# stations that never changed during the whole run: suspicious (offline?) or just very quiet
act[act.n_changes == 0][["name", "city", "min_bikes", "max_bikes", "capacity_official"]].head(20)

`last_reported` is a per-station timestamp. In the most recent snapshot, a very old value
(compared to `src_ts`) can mean the station has stopped reporting.

In [ ]:
last = snap[snap.src_ts == snap.src_ts.max()].merge(stations[["station_id", "name", "city"]], on="station_id")
last["stale_hours"] = (last.src_ts - last.last_reported) / 3600
print(last.stale_hours.describe().round(2))
last.sort_values("stale_hours", ascending=False)[["station_id", "name", "city", "bikes", "docks", "stale_hours"]].head(10)

Same idea over the whole run: the **worst** staleness per station, not just the latest snapshot. A healthy station reaches about 1.1 h between hourly heartbeats, so thresholds start at 2 h. `max_stale` above 2 h but `last_stale` back to normal means the station went silent and recovered.

In [ ]:
snap["stale_hours"] = (snap.src_ts - snap.last_reported) / 3600
stale = (
    snap.groupby("station_id")["stale_hours"].agg(max_stale="max", last_stale="last")
    .join(stations.set_index("station_id")[["name", "city"]])
)
print(stale.max_stale.quantile([.25, .5, .75, .95, .99]).round(2).to_string())
print({f">{h}h": int((stale.max_stale > h).sum()) for h in (2, 3, 6, 24)})
stale.sort_values("max_stale", ascending=False).head(20)

### NOTICE: stale stations still update appearently, Saint-Bon changed multiple times although the last reported date is constant (since 8 years ago)

## 5. How often do stations change?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
act.n_changes.plot.hist(bins=50, ax=ax[0])
ax[0].set(title="Changes per station (whole run)", xlabel="number of snapshots where bikes/docks changed")
act.bikes_moved.plot.hist(bins=50, ax=ax[1])
ax[1].set(title="Bikes moved per station", xlabel="sum of |change in bikes|")
plt.tight_layout()
plt.show()

In [ ]:
# only real one-minute steps count towards "% changing" (not a station's first snapshot, not the step across a gap)
steps = snap[snap.valid_step]
over_time = pd.DataFrame({
    "pct_stations_changing": steps.groupby(steps.time.dt.floor("30min"))["changed"].mean() * 100,
    "total_bikes": snap.groupby("time")["bikes"].sum().groupby(lambda t: t.floor("30min")).mean(),
}).asfreq("30min")  # bins with no snapshot (data gap) become NaN, so the lines break instead of bridging the gap
print("30 min bins with no data:", [f"{t:%a %d %b %H:%M}" for t in over_time.index[over_time.total_bikes.isna()]])
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
over_time.pct_stations_changing.plot(ax=ax[0], title="% of stations that changed between two snapshots (30 min bins)")
over_time.total_bikes.plot(ax=ax[1], title="Total bikes available in the system (30 min mean)")
plt.tight_layout()
plt.show()

In [ ]:
cols = ["name", "city", "n_changes", "bikes_moved", "max_bikes", "capacity_official"]
print("busiest stations:")
display(act.sort_values("bikes_moved", ascending=False)[cols].head(10))
print("quietest stations (but with at least one change):")
display(act[act.n_changes > 0].sort_values("n_changes")[cols].head(10))

### Four stations side by side

For each chosen station: bikes and free docks over time (left, lines break where data is missing), and the distribution of the change in bikes between two consecutive snapshots (right, log scale because most steps are 0). Edit `PICK` to look at other stations; several names match more than one station, so the ids are fixed here.

In [ ]:
PICK = {
    "Cité Universitaire": 1062874070,        # Jourdan - Cité Universitaire
    "Gare Saint-Lazare": 394404659,          # Gare Saint-Lazare - Cour du Havre
    "Vieille du Temple": 54000541,           # Vieille du Temple - Francs Bourgeois
    "Trocadéro": 213686471,                  # Eylau - Trocadéro
}
GAP = pd.Timedelta("5min")  # break the lines where data is missing (e.g. laptop offline)

fig, axes = plt.subplots(len(PICK), 2, figsize=(15, 3.6 * len(PICK)), gridspec_kw={"width_ratios": [3, 1.3]})
for (label, sid), (ax_t, ax_h) in zip(PICK.items(), axes):
    st = stations.set_index("station_id").loc[sid]
    one = snap[snap.station_id == sid].set_index("time")
    series = one[["bikes", "docks"]].astype(float)
    # steps-post carries each value forward to the next point, so the NaN must come right after the last
    # snapshot before the gap; placed just before the first snapshot after it, the line would run flat across the gap
    before_gap = (series.index.to_series().diff() > GAP).shift(-1, fill_value=False).to_numpy()
    gaps = pd.DataFrame(np.nan, index=series.index[before_gap] + pd.Timedelta("1s"), columns=series.columns)
    series = pd.concat([series, gaps]).sort_index()
    series.plot(ax=ax_t, drawstyle="steps-post", color=["tab:orange", "tab:blue"], lw=1.2)
    ax_t.set(title=f"{label}: {st['name']} (id {sid}, {int(st.capacity_official)} docks)", xlabel="", ylabel="bikes / free docks")
    ax_t.legend(["bikes", "free docks"], loc="best")

    # real one-minute steps only: the first snapshot and the step across a gap are not "0 change"
    diffs = one.loc[one.valid_step, "d_bikes"].astype(int)
    counts = diffs.value_counts().sort_index()
    ax_h.bar(counts.index, counts.values, color="tab:orange")
    ax_h.set_yscale("log")
    ax_h.set(title=f"change in bikes per snapshot (n changes = {int(one.changed.sum())})", xlabel="Δ bikes since previous snapshot", ylabel="number of snapshots (log)")
    ax_h.set_xticks(range(counts.index.min(), counts.index.max() + 1))
plt.tight_layout()
plt.show()

## 6. Map over time

One dot per station, average per 15 minute bin, coloured by share of slots rather than raw counts, so small and large
stations are on the same 0–100% scale. Two layers:
- **Free docks share** = docks / (bikes + docks) (blue, larger dot underneath)
- **Bikes share** = bikes / (bikes + docks) (orange, smaller dot on top)

The two shares add up to 100%, so each layer shows the same thing from the other side: click an entry in the legend to
switch a layer on or off. Offline stations (0 bikes and 0 docks) have no share and are left off the map.
Drag the slider (or press Play) to move through time. Hover a dot for the station name, counts and shares.

In [ ]:
BIN = "15min"
snap["bin"] = snap["time"].dt.floor(BIN)
binned = snap.groupby(["bin", "station_id"])[["bikes", "docks"]].mean().round()

meta = stations.set_index("station_id").sort_index()
# asfreq adds the bins with no snapshot (data gap) as all-NaN rows, so the slider shows them as "no data"
bikes_p = binned["bikes"].unstack("station_id").reindex(columns=meta.index).asfreq(BIN)
docks_p = binned["docks"].unstack("station_id").reindex(columns=meta.index).asfreq(BIN)
no_data = bikes_p.isna().all(axis=1).to_numpy()
print(f"{len(bikes_p)} time bins x {len(meta)} stations")
print("bins with no data:", [f"{t:%a %d %b %H:%M}" for t in bikes_p.index[no_data]])

In [ ]:
def marker(values, size, scale, title, x, opacity):
    return dict(
        color=values, size=size, colorscale=scale, cmin=0, cmax=1, opacity=opacity,
        showscale=True,
        colorbar=dict(title=title, x=x, len=0.4, thickness=12, tickformat=".0%"),
    )


labels = [f"{t:%a %d %b %H:%M}" + (" (no data)" if gap else "") for t, gap in zip(bikes_p.index, no_data)]
hover = ("%{text}<br>bikes: %{customdata[0]} (%{customdata[2]:.0%})"
         "<br>free docks: %{customdata[1]} (%{customdata[3]:.0%})<extra></extra>")


def traces_for(i):
    b, d = bikes_p.iloc[i].to_numpy(), docks_p.iloc[i].to_numpy()
    # stations seen in this bin (none in a data gap, so the map is empty there), minus offline ones
    # (0 bikes and 0 docks), whose shares are undefined
    keep = ~np.isnan(b) & (b + d > 0)
    m, b, d = meta[keep], b[keep], d[keep]
    b_share, d_share = b / (b + d), d / (b + d)
    cd = np.column_stack([b, d, b_share, d_share])
    return [
        go.Scattermap(lat=m.lat, lon=m.lon, mode="markers", name="Free docks share", text=m.name,
                      customdata=cd, hovertemplate=hover,
                      marker=marker(d_share, 15, "Blues", "Free docks /<br>(bikes + docks)", 1.0, 0.55)),
        go.Scattermap(lat=m.lat, lon=m.lon, mode="markers", name="Bikes share", text=m.name,
                      customdata=cd, hovertemplate=hover,
                      marker=marker(b_share, 7, "YlOrRd", "Bikes /<br>(bikes + docks)", 1.07, 0.95)),
    ]


frames = [go.Frame(name=labels[i], data=traces_for(i), traces=[0, 1]) for i in range(len(labels))]
slider_steps = [
    dict(method="animate", label=labels[i],
         args=[[labels[i]], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))])
    for i in range(len(labels))
]
fig = go.Figure(data=traces_for(0), frames=frames)
fig.update_layout(
    map=dict(style="open-street-map", center=dict(lat=meta.lat.mean(), lon=meta.lon.mean()), zoom=10.5),
    height=780, margin=dict(l=0, r=0, t=40, b=0),
    title="Velib stations over time: share of bikes / free docks (click the legend to toggle)",
    legend=dict(title="Layers (click to toggle)", x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
    sliders=[dict(active=0, steps=slider_steps, x=0.12, len=0.85, currentvalue=dict(prefix="Time: "))],
    updatemenus=[dict(
        type="buttons", showactive=False, x=0.0, y=0, xanchor="left", yanchor="top", pad=dict(t=40),
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=250, redraw=True), transition=dict(duration=0), fromcurrent=True)]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=0))]),
        ],
    )],
)
fig.show()

## Ideas for next steps
- Average the map by time of day across several days once more days are collected (a "typical Tuesday").
- Add the status flags (`is_installed`, `is_renting`, `is_returning`) to `snapshots.csv` to separate offline from quiet stations.
- If the data grows past what fits in memory, switch to Parquet or DuckDB/Postgres.

## 7. Binomial model with a time-of-day p(t)

For one station, bikes at minute t ~ Binomial(n_t, p(t)) with n_t = bikes + docks and
logit p(t) = β₀ + Σⱼ₌₁ᴷ [aⱼ cos(2πjτ) + bⱼ sin(2πjτ)], τ = time of day / 24h (so 23:59 joins 00:00).
The 2K + 1 parameters are fitted by maximum likelihood. Dots are the observed counts, lines the expected count n·p(t).

- **Equal weight per day:** some hours are covered by more days than others (e.g. at the start of the run 10:34–14:56 was seen on two days, the rest on one), so each
  snapshot is weighted by 1 / (number of days seen at that minute). Grey background = hours seen on only one day.
- **Offline snapshots** (0 bikes and 0 docks) are left out; minutes lost in a data gap are simply missing.
- **The band** is the binomial 5–95% spread given p(t). It ignores day-to-day variation, so it is narrower than what
  really happens; the cell prints how many observed counts actually fall inside it.
- **The likelihood treats every minute as independent**, but consecutive minutes are nearly identical, so the data hold
  far less information than the snapshot count suggests: don't build standard errors or tests on these fits.

With only one or two days per minute, a large K just traces those days. The helpers defined here are reused below.

In [ ]:
from scipy.optimize import minimize
from scipy.special import betaln, expit
from scipy.stats import binom

SID = 394404659  # Gare Saint-Lazare - Cour du Havre
KS = (2, 4)      # number of Fourier harmonics: 2K + 1 parameters
# one colour per day, kept away from the model lines (orange, blue)
DAY_COLORS = ["tab:green", "tab:purple", "tab:red", "tab:brown", "tab:pink", "tab:olive", "tab:cyan"]


def station_series(sid):
    # one station in time order, without offline snapshots (0 bikes and 0 docks)
    one = snap[(snap.station_id == sid) & (snap.bikes + snap.docks > 0)].sort_values("src_ts")
    t = one.time.dt
    minute = (t.hour * 60 + t.minute).to_numpy()              # minute of the day, 0..1439
    tau = (minute + t.second.to_numpy() / 60) / 1440          # time of day in [0, 1)
    day = t.date.to_numpy()
    k = one.bikes.to_numpy().astype(int)
    n = (one.bikes + one.docks).to_numpy().astype(int)
    # equal weight per day: a 5 min slot seen on 2 days gives each of its snapshots weight 1/2
    slot = minute // 5
    w = 1 / pd.Series(day).groupby(slot).nunique().reindex(slot).to_numpy()
    return minute, tau, day, k, n, w


def fourier(tau, K):
    cols = [np.ones_like(tau)]
    for j in range(1, K + 1):
        cols += [np.cos(2 * np.pi * j * tau), np.sin(2 * np.pi * j * tau)]
    return np.column_stack(cols)


def log_expit(z):
    # log(expit(z)) without underflow: p never rounds to exactly 0 or 1 inside the logs
    return -np.logaddexp(0, -z)


def nll_binom(beta, X, k, n, w):
    # weighted binomial negative log-likelihood, logit p(t) = X @ beta
    z = X @ beta
    return -np.sum(w * (k * log_expit(z) + (n - k) * log_expit(-z))) / w.sum()  # per snapshot: helps BFGS converge


def fit(nll, x0, *args):
    res = minimize(nll, x0, args=args, method="BFGS")
    if not res.success:
        print(f"warning: {nll.__name__} stopped before full convergence ({res.message})")
    return res.x


def equal_day_mean(per_day):
    # rows = days; average the days that have data, each with the same weight
    n_days = (~np.isnan(per_day)).sum(axis=0)
    return np.where(n_days > 0, np.nansum(per_day, axis=0) / np.maximum(n_days, 1), np.nan), n_days


def ratio(ks, ns):
    return np.where(ns > 0, ks / np.where(ns > 0, ns, 1), np.nan)


def sliding_mean(minute, day, k, n, win):
    # centred sliding Σk / Σn over time of day (wrapping around midnight), per day, then averaged over days
    h = win // 2
    per_day = []
    for d in np.unique(day):
        m = day == d
        ks, ns = (np.bincount(minute[m], weights=v[m], minlength=1440) for v in (k, n))
        ks, ns = (np.convolve(np.r_[v[len(v) - h:], v, v[:h]], np.ones(win), mode="valid") for v in (ks, ns))
        per_day.append(ratio(ks, ns))
    return equal_day_mean(np.array(per_day))


def binned_mean(minute, day, k, n, width):
    # Σk / Σn per time-of-day bin, per day, then averaged over days
    per_day = []
    for d in np.unique(day):
        m = day == d
        ks, ns = (np.bincount(minute[m] // width, weights=v[m], minlength=1440 // width) for v in (k, n))
        per_day.append(ratio(ks, ns))
    return equal_day_mean(np.array(per_day))


def scatter_days(ax, tau, day, k, alpha=0.6):
    for i, d in enumerate(sorted(set(day))):
        m = day == d
        ax.scatter(tau[m] * 24, k[m], s=8, marker="os^D"[i % 4], color=DAY_COLORS[i % len(DAY_COLORS)],
                   alpha=alpha, lw=0, label=f"observed bikes, {d:%a %d %b}")


def shade_single_day(ax, n_days_min):
    ax.fill_between((np.arange(1440) + 0.5) / 60, 0, 1, where=n_days_min == 1, transform=ax.get_xaxis_transform(),
                    color="0.93", lw=0, label="only 1 day of data")


def time_axis(ax):
    ax.set(xlim=(0, 24), xlabel="time of day (Paris)")
    ax.set_xticks(range(0, 25, 3), [f"{h:02d}:00" for h in range(0, 25, 3)])
    ax.grid(axis="y", alpha=0.3)


minute, tau, day, k, n, w = station_series(SID)
n_days_min = sliding_mean(minute, day, k, n, 5)[1]  # number of days with data around each minute (±2 min)
grid = np.linspace(0, 1, 1441)
n_typ = int(np.median(n))
beta_fit, p_fit = {}, {}
for K in KS:
    beta_fit[K] = fit(nll_binom, np.zeros(2 * K + 1), fourier(tau, K), k, n, w)
    p_fit[K] = expit(fourier(grid, K) @ beta_fit[K])

K_band = max(KS)
p_obs = expit(fourier(tau, K_band) @ beta_fit[K_band])
inside = (k >= binom.ppf(0.05, n, p_obs)) & (k <= binom.ppf(0.95, n, p_obs))
print(f"binomial 5–95% band (K={K_band}, each snapshot's own n) contains {inside.mean():.1%} of observed counts "
      "(about 90% if the binomial were right)")

fig, ax = plt.subplots(figsize=(13, 5.5))
shade_single_day(ax, n_days_min)
ax.fill_between(grid * 24, binom.ppf(0.05, n_typ, p_fit[K_band]), binom.ppf(0.95, n_typ, p_fit[K_band]),
                color="tab:blue", alpha=0.15, lw=0,
                label=f"K={K_band} binomial 5–95% spread given p(t) (ignores day-to-day variation)")
scatter_days(ax, tau, day, k)
for K, color in zip(KS, ["tab:orange", "tab:blue"]):
    ax.plot(grid * 24, n_typ * p_fit[K], color=color, lw=2, label=f"model K={K} (mean n·p(t), {2 * K + 1} params)")
time_axis(ax)
ax.set(ylim=(0, n.max()), ylabel=f"available bikes (n = bikes + docks ≈ {n_typ})",
       title=f"{stations.set_index('station_id').loc[SID, 'name']} (id {SID}): Binomial(n, p(t)), logit p(t) = Fourier series")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

### Nonparametric alternative: p per 5 minute bin

No shape assumed: the day is cut into 5 minute bins, p for a bin is Σ bikes / Σ (bikes + docks) within each day, and the
days are then averaged with equal weight (288 free parameters). It needs many days to be stable; with one or two days it
mostly reproduces them. Uses the helpers and variables from the cell above, so run that one first.

In [ ]:
BIN_MIN = 5  # width of the time-of-day bins, in minutes

p_bin, n_days_bin = binned_mean(minute, day, k, n, BIN_MIN)
edges = np.arange(len(p_bin) + 1) * BIN_MIN / 60
print(f"{len(p_bin)} bins of {BIN_MIN} min | bins with no data: {int(np.isnan(p_bin).sum())} | "
      f"days per bin: {pd.Series(n_days_bin).value_counts().sort_index().to_dict()}")

fig, ax = plt.subplots(figsize=(13, 5.5))
shade_single_day(ax, n_days_min)
scatter_days(ax, tau, day, k, alpha=0.5)
ax.stairs(n_typ * p_bin, edges, baseline=None, color="black", lw=1.5, label=f"{BIN_MIN} min bins, equal weight per day")
for K, color in zip(KS, ["tab:orange", "tab:blue"]):
    ax.plot(grid * 24, n_typ * p_fit[K], color=color, lw=2, label=f"Fourier K={K}")
time_axis(ax)
ax.set(ylim=(0, n.max()), ylabel=f"available bikes (n ≈ {n_typ})",
       title=f"{stations.set_index('station_id').loc[SID, 'name']} (id {SID}): {BIN_MIN} min bins vs Fourier p(t)")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

### Sliding average

Same idea as the bins, but the window (`WIN_MIN` minutes wide, centred) moves one minute at a time: p at minute m is
Σ bikes / Σ (bikes + docks) over the snapshots in [m − WIN_MIN // 2, m + WIN_MIN // 2] within each day, and the days are
then averaged with equal weight. Each minute borrows from its neighbours, so the curve is smoother than the bins,
without assuming a shape like the Fourier series does. Averaging per day first avoids a jump where a second day's data
enters the window. Uses the helpers and variables from the cells above.

In [ ]:
WIN_MIN = 61  # width of the sliding window, in minutes (odd, centred on each minute)

p_roll, n_days_roll = sliding_mean(minute, day, k, n, WIN_MIN)
print(f"minutes with no data in their {WIN_MIN} min window: {int(np.isnan(p_roll).sum())}")

fig, ax = plt.subplots(figsize=(13, 5.5))
shade_single_day(ax, n_days_min)
scatter_days(ax, tau, day, k, alpha=0.5)
ax.stairs(n_typ * p_bin, edges, baseline=None, color="0.6", lw=1, label=f"{BIN_MIN} min bins")
ax.plot((np.arange(1440) + 0.5) / 60, n_typ * p_roll, color="black", lw=2, label=f"{WIN_MIN} min sliding average")
ax.plot(grid * 24, n_typ * p_fit[max(KS)], color="tab:blue", lw=2, label=f"Fourier K={max(KS)}")
time_axis(ax)
ax.set(ylim=(0, n.max()), ylabel=f"available bikes (n ≈ {n_typ})",
       title=f"{stations.set_index('station_id').loc[SID, 'name']} (id {SID}): {WIN_MIN} min sliding average vs Fourier p(t)")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

### P(bike available) and P(dock available) at a given time

What we want to predict is the probability of finding **at least one bike** (to start a trip) or **at least one free
dock** (to end one) at, say, 10:00. With k bikes out of n slots:
- P(bike available) = 1 − P(k = 0) and P(dock available) = 1 − P(k = n), since "at least one" is the opposite of "none".
  Three ways to get P(k = 0) and P(k = n):
  - **binomial:** fit p(t) on the counts, then P(0) = (1 − p)ⁿ and P(full) = pⁿ
  - **beta-binomial:** same, but each snapshot draws its own p around p(t), giving B(a, b + n)/B(a, b) and B(a + n, b)/B(a, b).
    This only absorbs scatter of single snapshots; it cannot represent a whole day sitting above or below the curve
    (that needs logit p_d(t) = f(t) + u_d and many days). When its fitted φ is large it is practically the binomial.
  - **logistic:** skip the counts and model the event itself (1{0 bikes} or 1{0 docks}) with the same Fourier terms.
    Skipped when the station is never (or always) empty / full.
- Black lines: observed share of snapshots with a bike (or a free dock) available, in a sliding window (equal weight per day).
  Grey background: hours seen on only one day.

Two stations: Jourdan - Cité Universitaire is often empty, Jussieu - Fossés Saint-Bernard is often full (no station here is
often both).

**No ranking of the models is possible with this data.** Scoring them on the data they were fitted to favours the
logistic model by construction, almost every event was seen on a single day, and events come in a few long runs, so
the effective sample is tiny. A fair comparison needs several days and scoring on held-out days.

In [ ]:
def nll_betabin(theta, X, k, n, w):
    # each snapshot's p ~ Beta(p(t)·phi, (1 - p(t))·phi), independently (not a per-day effect)
    p, phi = expit(X @ theta[:-1]), np.exp(theta[-1])
    return -np.sum(w * (betaln(k + p * phi, n - k + (1 - p) * phi) - betaln(p * phi, (1 - p) * phi))) / w.sum()


def nll_logistic(beta, X, y, w):
    # logistic regression of a yes/no event y
    z = X @ beta
    return -np.sum(w * (y * log_expit(z) + (1 - y) * log_expit(-z))) / w.sum()


def p_available(sid, K=4, win=61, at=None):
    # P(bike available) and P(dock available) over the day for one station: three models vs what happened
    minute_s, tau_s, day_s, k_s, n_s, w_s = station_series(sid)
    X, G = fourier(tau_s, K), fourier(grid, K)
    n_s_typ = int(np.median(n_s))
    b_bin = fit(nll_binom, np.zeros(2 * K + 1), X, k_s, n_s, w_s)
    th_bb = fit(nll_betabin, np.r_[b_bin, 2.0], X, k_s, n_s, w_s)
    p_bin_g, p_bb_g, phi = expit(G @ b_bin), expit(G @ th_bb[:-1]), np.exp(th_bb[-1])
    a, b = p_bb_g * phi, (1 - p_bb_g) * phi
    n_days_s = sliding_mean(minute_s, day_s, k_s, n_s, 5)[1]
    name = stations.set_index("station_id").loc[sid, "name"]
    x_min = (np.arange(1440) + 0.5) / 60

    # P(available) = 1 - P(none): none = 0 bikes (k = 0) or 0 free docks (k = n)
    events = {  # what we want: (snapshot had none, binomial P(none), beta-binomial P(none))
        "bike available": (k_s == 0, (1 - p_bin_g) ** n_s_typ, np.exp(betaln(a, b + n_s_typ) - betaln(a, b))),
        "dock available": (k_s == n_s, p_bin_g ** n_s_typ, np.exp(betaln(a + n_s_typ, b) - betaln(a, b))),
    }
    print(f"{name} (id {sid}): {len(k_s)} snapshots, n ≈ {n_s_typ}, beta-binomial φ = {phi:.0f}")

    fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
    for ax in axes:
        shade_single_day(ax, n_days_s)
    scatter_days(axes[0], tau_s, day_s, k_s, alpha=0.5)
    axes[0].plot(x_min, n_s_typ * sliding_mean(minute_s, day_s, k_s, n_s, win)[0], color="black", lw=2,
                 label=f"{win} min sliding average")
    axes[0].plot(grid * 24, n_s_typ * p_bin_g, color="tab:blue", lw=2, label=f"binomial Fourier K={K} (mean n·p)")
    axes[0].plot(grid * 24, n_s_typ * p_bb_g, color="tab:orange", lw=2, ls="--", label=f"beta-binomial K={K} (mean)")
    axes[0].set(ylim=(0, n_s.max()), ylabel=f"available bikes (n ≈ {n_s_typ})", title=f"{name} (id {sid})")
    axes[0].grid(axis="y", alpha=0.3)
    axes[0].legend(loc="best", frameon=False)

    for ax, (event, (y, q_bin, q_bb)) in zip(axes[1:], events.items()):
        y = y.astype(float)  # 1 = none available
        runs = int(np.sum(np.diff(np.r_[0.0, y]) == 1))
        seen_twice = n_days_s[minute_s] >= 2
        print(f"  {event}: yes in {1 - y.mean():.1%} of snapshots; the {y.mean():.1%} without come in {runs} separate runs "
              f"({int(y[seen_twice].sum())} of those snapshots at minutes seen on 2+ days)")
        ax.plot(x_min, 1 - sliding_mean(minute_s, day_s, y, np.ones_like(y), win)[0], color="black", lw=2,
                label=f"observed share ({win} min window)")
        ax.plot(grid * 24, 1 - q_bin, color="tab:blue", lw=2, label=f"binomial K={K}")
        ax.plot(grid * 24, 1 - q_bb, color="tab:orange", lw=2, ls="--", label=f"beta-binomial K={K}")
        if 0 < y.sum() < len(y):
            q_log = expit(G @ fit(nll_logistic, np.zeros(2 * K + 1), X, y, w_s))
            ax.plot(grid * 24, 1 - q_log, color="tab:red", lw=2, label=f"logistic on 1{{{event}}} K={K}")
        else:
            q_log = None
        if at is not None:
            i = int(round(at * 60))  # grid has one point per minute
            vals = [f"binomial {1 - q_bin[i]:.1%}", f"beta-binomial {1 - q_bb[i]:.1%}"]
            vals += [f"logistic {1 - q_log[i]:.1%}"] if q_log is not None else []
            print(f"    P({event} at {int(at):02d}:{int(at % 1 * 60):02d}): " + ", ".join(vals))
        ax.set(ylim=(-0.02, 1.02), ylabel=f"P({event})")
        ax.grid(axis="y", alpha=0.3)
        ax.legend(loc="best", frameon=False)
    time_axis(axes[-1])
    plt.tight_layout()
    plt.show()


p_available(1062874070, at=10)  # Jourdan - Cité Universitaire: often empty
p_available(54000544, at=10)    # Jussieu - Fossés Saint-Bernard: often full
p_available(213686471, at=12)   # Trocadero